In [ ]:
#匯入套件與設定環境
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import requests
from PIL import Image
from google.colab import files
from io import BytesIO
from ultralytics import YOLO

In [ ]:
#讀取圖片
def get_image_from_url(url_list):

  image_list = []

  for url in url_list:
    response = requests.get(url)
    image = Image.open(BytesIO(response.content))
    image_list.append(image)

  return image_list

In [ ]:
#調整大小
def resize_image(image, size):

  width, height = image.size
  aspect_ratio = 4 / 3

  #計算4:3的裁剪範圍
  if width / height > aspect_ratio:
    #圖片太寬，以高度為基準計算新寬度
    new_width = int(height * aspect_ratio)
    new_height = height
  else:
    #圖片太高，以寬度為基準計算新高度
    new_width = width
    new_height = int(width / aspect_ratio)

  #計算裁剪的範圍（從中心開始）
  left = (width - new_width) // 2
  top = (height - new_height) // 2
  right = left + new_width
  bottom = top + new_height

  #裁剪4:3區域
  img_cropped = image.crop((left, top, right, bottom))

  #調整大小
  img_resized = img_cropped.resize(size, Image.LANCZOS)

  return img_resized

In [ ]:
#圖片分割
def split_image(image, block_size):

  width, height = image.size
  stride = block_size // 2
  num_row = height // block_size[1]
  num_col = width // block_size[0]
  block_list = []

  for row in range(num_row):
    for col in range(num_col):
      left = col * stride  #左上角x座標
      top = row * stride   #左上角y座標
      right = left + block_size[0]  #右下角x座標
      bottom = top + block_size[1]  #右下角y座標

      #裁剪圖片
      block = image.crop((left, top, right, bottom))
      block_list.append(block)

  return block_list

In [ ]:
image_size = (2560, 1920)
block size = (640, 640)